# Final k-NN Model

After comparing the different settings in the previous experiments, I used the best k values from V8 to train the final models. This time, I used all the labeled training samples from both datasets.

First, I imported the libraries and functions I'll need.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data_loader import load_task
from src.models.knn import KNNClassifier, euclidean_distance

image_v8_summary = pd.read_csv(project_root / "results" / "image" / "knn_v8_summary.csv")
sentiment_v8_summary = pd.read_csv(project_root / "results" / "sentiment" / "knn_v8_summary.csv")
image_v8_best = image_v8_summary.sort_values(
    ["validation_accuracy", "k"], ascending=[False, True]
).iloc[0]
sentiment_v8_best = sentiment_v8_summary.sort_values(
    ["validation_accuracy", "k"], ascending=[False, True]
).iloc[0]

for dataset, summary, best, expected_k, expected_accuracy in [
    ("image", image_v8_summary, image_v8_best, 1, 0.39625),
    ("sentiment", sentiment_v8_summary, sentiment_v8_best, 20, 0.5629166666666666),
]:
    assert best["k"] == expected_k
    assert abs(best["validation_accuracy"] - expected_accuracy) <= 1e-15
    assert (summary["distance_metric"] == "euclidean").all()
    assert (summary["voting_method"] == "majority").all()
    assert (summary["normalized"] == False).all()
    print(f"V8 settings confirmed: {dataset}, k={expected_k}, Euclidean, majority, original features")


V8 settings confirmed: image, k=1, Euclidean, majority, original features
V8 settings confirmed: sentiment, k=20, Euclidean, majority, original features


I started by training the image model with all 20000 labeled samples.

In [2]:
image_X_train, image_y_train, image_X_test, image_test_ids = load_task("image")
assert image_X_train.shape == (20000, 500)
assert image_y_train.shape == (20000,)
assert image_X_test.shape == (5000, 500)
assert len(image_test_ids) == 5000
assert pd.api.types.is_integer_dtype(image_y_train)
assert ((image_y_train >= 0) & (image_y_train <= 49)).all()

image_clf = KNNClassifier(k=1, distance_fn=euclidean_distance, voting_method="majority")
image_clf.fit(image_X_train, image_y_train)
print("Image: predicting test labels using all 20000 labeled training samples")
start = perf_counter()
image_test_predictions = image_clf.predict(image_X_test)
image_prediction_time = perf_counter() - start
assert len(image_test_predictions) == 5000
print(f"Image: training={len(image_X_train)}, test={len(image_X_test)}, k=1, distance=euclidean, voting=majority")
print(f"Prediction time: {image_prediction_time:.6f} s")


Image: predicting test labels using all 20000 labeled training samples
Image: training=20000, test=5000, k=1, distance=euclidean, voting=majority
Prediction time: 541.209908 s


Then I did the same thing for sentiment, using all 12000 labeled samples.

In [3]:
sentiment_X_train, sentiment_y_train, sentiment_X_test, sentiment_test_ids = load_task("sentiment")
assert sentiment_X_train.shape == (12000, 768)
assert sentiment_y_train.shape == (12000,)
assert sentiment_X_test.shape == (5000, 768)
assert len(sentiment_test_ids) == 5000
assert pd.api.types.is_integer_dtype(sentiment_y_train)
assert ((sentiment_y_train >= 0) & (sentiment_y_train <= 2)).all()

sentiment_clf = KNNClassifier(k=20, distance_fn=euclidean_distance, voting_method="majority")
sentiment_clf.fit(sentiment_X_train, sentiment_y_train)
print("Sentiment: predicting test labels using all 12000 labeled training samples")
start = perf_counter()
sentiment_test_predictions = sentiment_clf.predict(sentiment_X_test)
sentiment_prediction_time = perf_counter() - start
assert len(sentiment_test_predictions) == 5000
print(f"Sentiment: training={len(sentiment_X_train)}, test={len(sentiment_X_test)}, k=20, distance=euclidean, voting=majority")
print(f"Prediction time: {sentiment_prediction_time:.6f} s")


Sentiment: predicting test labels using all 12000 labeled training samples
Sentiment: training=12000, test=5000, k=20, distance=euclidean, voting=majority
Prediction time: 315.387700 s


Finally, I combined the predictions into one Kaggle submission file.

In [4]:
image_predictions = pd.DataFrame({"id": image_test_ids, "label": image_test_predictions})
sentiment_predictions = pd.DataFrame({"id": sentiment_test_ids, "label": sentiment_test_predictions})
assert len(image_predictions) == 5000 and len(sentiment_predictions) == 5000
assert image_predictions["label"].between(0, 49).all()
assert sentiment_predictions["label"].between(0, 2).all()

predictions = pd.concat([image_predictions, sentiment_predictions], ignore_index=True)
sample_submission = pd.read_csv(project_root / "data" / "raw" / "sample_submission.csv")
assert len(predictions) == 10000 and len(sample_submission) == 10000
assert predictions["id"].is_unique and sample_submission["id"].is_unique
assert set(predictions["id"]) == set(sample_submission["id"])

submission = sample_submission[["id"]].merge(predictions, on="id", how="left", sort=False)
assert submission.columns.tolist() == ["id", "label"]
assert submission["id"].tolist() == sample_submission["id"].tolist()
assert not submission.isna().any().any()
assert pd.api.types.is_integer_dtype(submission["label"])

submission_path = project_root / "submission" / "knn" / "submission_final_knn.csv"
submission.to_csv(submission_path, index=False)
saved_submission = pd.read_csv(submission_path)
assert saved_submission.columns.tolist() == ["id", "label"]
assert len(saved_submission) == 10000
assert saved_submission["id"].tolist() == sample_submission["id"].tolist()
assert saved_submission["id"].is_unique
assert not saved_submission.isna().any().any()
assert pd.api.types.is_integer_dtype(saved_submission["label"])
assert saved_submission["id"].isin(image_test_ids).sum() == 5000
assert saved_submission["id"].isin(sentiment_test_ids).sum() == 5000
assert saved_submission.loc[saved_submission["id"].isin(image_test_ids), "label"].between(0, 49).all()
assert saved_submission.loc[saved_submission["id"].isin(sentiment_test_ids), "label"].between(0, 2).all()
pd.testing.assert_frame_equal(saved_submission, submission, check_dtype=False)
print("Saved and checked 10000 predictions in", submission_path)


Saved and checked 10000 predictions in C:\Users\user\OneDrive\Documentos\Fall2026\COMP472\team_project\COMP472\submission\knn\submission_final_knn.csv


I compared the final predictions with V8 to see how much they changed.

In [5]:
v8_submission = pd.read_csv(project_root / "submission" / "knn" / "submission_v8_knn.csv")
assert v8_submission.columns.tolist() == ["id", "label"]
assert len(v8_submission) == 10000 and v8_submission["id"].is_unique
assert set(v8_submission["id"]) == set(saved_submission["id"])
comparison = saved_submission.merge(
    v8_submission, on="id", how="left", sort=False,
    suffixes=("_final", "_v8"), validate="one_to_one"
)
changed = comparison["label_final"] != comparison["label_v8"]
changed_predictions = int(changed.sum())
changed_image_predictions = int((changed & comparison["id"].isin(image_test_ids)).sum())
changed_sentiment_predictions = int((changed & comparison["id"].isin(sentiment_test_ids)).sum())
assert changed_predictions == changed_image_predictions + changed_sentiment_predictions
print(f"Predictions identical to V8: {changed_predictions == 0}")
print(f"Changed predictions compared with V8: {changed_predictions}")
print(f"Changed image predictions: {changed_image_predictions}")
print(f"Changed sentiment predictions: {changed_sentiment_predictions}")


Predictions identical to V8: False
Changed predictions compared with V8: 2911
Changed image predictions: 2176
Changed sentiment predictions: 735


I saved the settings and prediction times. The V8 accuracies are the earlier held-out results.

In [6]:
final_rows = []
for dataset, model, train_features, test_features, previous, prediction_time in [
    ("image", image_clf, image_X_train, image_X_test, image_v8_best, image_prediction_time),
    ("sentiment", sentiment_clf, sentiment_X_train, sentiment_X_test, sentiment_v8_best, sentiment_prediction_time),
]:
    final_rows.append({
        "dataset": dataset,
        "version": "Final",
        "k": model.k,
        "distance_metric": "euclidean",
        "voting_method": "majority",
        "normalized": False,
        "training_samples": len(train_features),
        "test_samples": len(test_features),
        "selected_using": "V8 validation",
        "v8_validation_accuracy": previous["validation_accuracy"],
        "prediction_time_seconds": prediction_time,
    })
knn_result_dir = project_root / "results" / "knn"
final_summary = pd.DataFrame(final_rows)
final_summary.to_csv(knn_result_dir / "knn_final_summary.csv", index=False)
print(final_summary.to_string(index=False))


  dataset version  k distance_metric voting_method  normalized  training_samples  test_samples selected_using  v8_validation_accuracy  prediction_time_seconds
    image   Final  1       euclidean      majority       False             20000          5000  V8 validation                0.396250               541.209908
sentiment   Final 20       euclidean      majority       False             12000          5000  V8 validation                0.562917               315.387700


In [7]:
tracker_path = knn_result_dir / "kaggle_scores.csv"
tracker = pd.read_csv(tracker_path, dtype=str, keep_default_na=False)
assert tracker["version"].is_unique
final_row = {
    "version": "Final",
    "image_validation_accuracy": "",
    "sentiment_validation_accuracy": "",
    "kaggle_public_score": "",
    "submitted_to_kaggle": False,
    "submission_filename": submission_path.name,
    "notes": "Final k-NN trained on all labeled samples; image k=1 (20000 samples), sentiment k=20 (12000 samples). Parameters selected using V8 validation.",
}
matching = tracker["version"] == "Final"
if matching.any():
    # Preserve any real Kaggle score and submission status on reruns.
    for column in ("image_validation_accuracy", "sentiment_validation_accuracy", "submission_filename", "notes"):
        tracker.loc[matching, column] = final_row[column]
else:
    tracker = pd.concat([tracker, pd.DataFrame([final_row])], ignore_index=True)
assert tracker["version"].is_unique
tracker.to_csv(tracker_path, index=False)
print(tracker.to_string(index=False))


version image_validation_accuracy sentiment_validation_accuracy kaggle_public_score submitted_to_kaggle      submission_filename                                                                                                                                          notes
     V1                      0.21                         0.575                                   False                                                                                                                             k=3 and 1,000 training samples per dataset.
     V2                     0.215                         0.575               0.329                True    submission_v2_knn.csv                                                                                       Tested k=1, 3, 5, 10, 20; best image k=1, sentiment k=3.
     V3                     0.215                         0.575                                   False    submission_v3_knn.csv                                                Compared